# Companion threads — background work while the caller talks

A workflow normally runs **one** thread: a node executes, an edge is taken, the next node
executes, and when a node waits for the caller everything stops until they speak.

A **companion thread** breaks that. Flag a direct edge with `CompanionThreadConfig` and the
graph forks: the main thread carries on the conversation, and the companion keeps executing —
and keeps emitting events — while the main thread is parked.

The motivating case is the one built below: a caller asks "are my lab results back yet?" while
a companion polls the lab system every few seconds.

This is an **authoring-time** feature. There is no `client.companion_threads` resource; you
configure it on an edge and the runtime does the rest.

**See also:** [Companion threads guide](../docs/guides/companion_threads.md) ·
[`wf_examples/wf_example_progression_24.md`](../wf_examples/wf_example_progression_24.md)

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

## 1. Build the graph

Five nodes. The shape matters more than the contents:

```
        ┌──────────────► Poll Lab Results  (companion thread "labpoll")
        │                       │
     Entry                      ▼
        │                 Result Landed
        └──────────────► Chat While Waiting ──[waiting conditional]──► Deliver Result
```

`Entry` exists **only** to fan out. It has to: a node with outgoing direct edges cannot also
carry an evaluate-while-waiting conditional edge, so the fork cannot live on the chat node.
More on that rule in section 3.

In [ ]:
import _bootstrap  # noqa: F401

import interactly_configs as ic
from interactly import AsyncWorkflowClient, WorkflowCommand

client = AsyncWorkflowClient()

# A stand-in for "call the lab's API and see whether the result is back".
# It returns a FLAT integer — dotted access into a dict-valued runtime variable is not
# interpolated, so keep tool results flat when a condition needs to read them.
POLL_CODE = """
def poll_lab_results(attempts_so_far):
    try:
        n = int(attempts_so_far)
    except (TypeError, ValueError):
        n = 0
    return n + 1
"""

llm = ic.LLMGroupConfig(llms=[ic.OpenAILLMConfig(model=ic.OPENAIModel.GPT_4_1_MINI, max_tokens=120)])

entry = ic.NoOpNodeConfig(name="Entry", is_start=True, note="Fork point only.")

poller = ic.ToolNodeConfig(
    name="Poll Lab Results",
    tool_arguments={"attempts_so_far": "[[lab_attempts]]"},
    result_runtime_variable_name="lab_attempts",
    # A bound is REQUIRED — see notebook 20 and the self-loops guide.
    self_loop_config=ic.SelfLoopConfig(enabled=True, max_retries=8, expiry_time=60, time_between_retries=3),
    tool_config=ic.InlinePythonToolConfig(
        name="poll_lab_results",
        description="Check whether the lab result is available yet",
        signature="Given attempts so far, returns the new attempt count.",
        args_schema={"type": "object",
                     "properties": {"attempts_so_far": {"type": "number", "description": "Attempts so far"}},
                     "required": ["attempts_so_far"]},
        code=POLL_CODE,
    ),
)

# A companion must terminate within its turn, so it needs somewhere to land.
landed = ic.NoOpNodeConfig(name="Result Landed", output_runtime_variable_name="lab_result_ready")

chat = ic.SayLLMNodeConfig(
    name="Chat While Waiting",
    wait_for_user_message=True,
    self_loop=True,
    main_response_config=ic.PromptConfig(
        prompt="Keep the caller company while their lab results are fetched. ONE short sentence. "
               "Never claim the results have arrived."),
    llms_config=llm,
)

deliver = ic.SayStaticMessageNodeConfig(
    name="Deliver Result",
    static_messages_config=ic.StaticMessagesConfig(
        static_messages=["Good news — your lab results just came through: A1C 5.4%, within range."]),
)

print("nodes built")

## 2. Fork the thread

A companion is a **direct edge with `companion_thread_config` set**. Two rules are visible here:

- **Exactly one main edge.** If a node has several outgoing direct edges, exactly one must have
  `is_companion_thread=False` and all the rest `True`. Two "main" edges would mean two
  conversations with one caller.
- **Name the thread if anything needs to read it.** Without `thread_id` the companion still runs,
  but the server mints a uuid at fork time and the thread is not addressable — which would make
  the waiting condition below impossible to write.

In [ ]:
edges = [
    # The MAIN thread.
    ic.DirectEdgeConfig(source_node_logical_id=entry.logical_id,
                        destination_node_logical_id=chat.logical_id),

    # The COMPANION fork.
    ic.DirectEdgeConfig(
        source_node_logical_id=entry.logical_id,
        destination_node_logical_id=poller.logical_id,
        companion_thread_config=ic.CompanionThreadConfig(is_companion_thread=True, thread_id="labpoll"),
    ),

    # The poller's own exit, so it stops as soon as the result lands.
    ic.ConditionalEdgeConfig(
        source_node_logical_id=poller.logical_id,
        destination_node_logical_id=landed.logical_id,
        condition=ic.ConditionConfig(condition_expression="[[lab_attempts]] >= 3"),
    ),

    # Advance the CONVERSATION when the COMPANION's variable changes — no user message needed.
    # Notebook 20 covers this edge in detail.
    ic.ConditionalEdgeConfig(
        source_node_logical_id=chat.logical_id,
        destination_node_logical_id=deliver.logical_id,
        condition=ic.ConditionConfig(condition_expression="[[thread_labpoll.lab_attempts]] >= 3"),
        evaluate_while_waiting_config=ic.EvaluateWhileWaitingConfig(
            enabled=True, trigger_node_logical_ids=[poller.logical_id]),
    ),
]

# Four defensive accessors read the nested config safely on ANY edge, including edges
# stored before the feature existed.
for e in edges:
    print(f"{e.type:12s} companion={ic.edge_is_companion(e)!s:5s} "
          f"thread_id={ic.edge_companion_thread_id(e)!s:10s} "
          f"waits={ic.edge_evaluates_while_waiting(e)}")

In [ ]:
config = ic.WorkflowConfigFullyHydrated(
    workflow_config=ic.WorkflowConfig(name="NB19: Companion threads"),
    node_configs=[entry, poller, landed, chat, deliver],
    edge_configs=edges,
)

workflow = await client.workflows.create_from_config(config, name="NB19: Companion threads")
WORKFLOW_ID = workflow.id
print("Created", WORKFLOW_ID)

## 3. The rule that catches people out

**A node with outgoing direct edges cannot also carry an evaluate-while-waiting conditional edge.**

Direct edges take precedence on the normal transition path, so such an edge would fire *only* in
the background and never after a node execution — divergent behaviour from one config. The server
refuses it at save time rather than letting it misbehave quietly.

In practice this means: **fork the companion upstream of the node that waits.** Below is the same
graph with the fork moved onto `chat`, which is the mistake this rule exists to catch.

In [ ]:
from interactly import BadRequestError

bad_edges = [
    ic.DirectEdgeConfig(source_node_logical_id=entry.logical_id,
                        destination_node_logical_id=chat.logical_id),
    # WRONG: the fork now leaves the node that also carries the waiting edge.
    ic.DirectEdgeConfig(
        source_node_logical_id=chat.logical_id,
        destination_node_logical_id=poller.logical_id,
        companion_thread_config=ic.CompanionThreadConfig(is_companion_thread=True, thread_id="labpoll2"),
    ),
    ic.ConditionalEdgeConfig(
        source_node_logical_id=chat.logical_id,
        destination_node_logical_id=deliver.logical_id,
        condition=ic.ConditionConfig(condition_expression="[[thread_labpoll2.lab_attempts]] >= 3"),
        evaluate_while_waiting_config=ic.EvaluateWhileWaitingConfig(
            enabled=True, trigger_node_logical_ids=[poller.logical_id]),
    ),
]

bad_config = ic.WorkflowConfigFullyHydrated(
    workflow_config=ic.WorkflowConfig(name="NB19: rejected"),
    node_configs=[entry, poller, landed, chat, deliver],
    edge_configs=bad_edges,
)

try:
    await client.workflows.create_from_config(bad_config, name="NB19: rejected")
    print("unexpectedly accepted — clean this up!")
except BadRequestError as exc:
    print("Rejected, as it should be:\n")
    print(exc)

### The other three graph rules

All validated at save time and rejected with **400**:

| # | Rule | Why |
|---|---|---|
| 1 | Exactly one main edge per node | Two "main" edges would mean two conversations with one caller. |
| 2 | Companions must be non-interactive | A companion never receives user messages, so a `wait_for_user_message=True` node inside one would park forever. |
| 3 | No nested forks | Companions fork from the main thread only. |
| 4 | No convergence back onto the main thread | A node reachable from both would be classified globally as a companion node, stripping its global-node fan-in even when the *main* thread executes it. Communicate through variables instead. |

`thread_id` has rules of its own: `[A-Za-z0-9_-]` only (no dots — they separate the id from the
variable name), must not contain `_companion_`, must not be `"0"` (reserved for the main thread),
and must be unique across the whole workflow.

## 4. Drive it over WebSocket

The WebSocket driver pumps background work itself, so there is nothing to do but read the stream.

**One thing changes** once companions are in play, and it is the single most common bug here:

| Event | Means |
|---|---|
| `end_workflow_iteration` | One turn's accounting is done. Companions may still be running. |
| `workflow_ready_for_input` | Every non-companion thread has settled. **Send the next message.** |
| `end_workflow` | Every thread has ended, companions included. |

A loop that breaks on `end_workflow_iteration` will race or cut off mid-turn. Break on
`is_ready_for_input()` instead.

In [ ]:
timeline = []

async with client.runs.stream(workflow_id=WORKFLOW_ID, command=WorkflowCommand.START) as stream:
    async for event in stream:
        timeline.append((event.thread_reference_id, event.type))

        if event.type == "assistant_response":
            print(f"🤖 [{event.thread_reference_id}] {event.output}")
        elif event.type == "companion_step_boundary":
            print("   ⏳ one companion step finished")
        elif event.type == "waiting_condition_matched":
            print("   ⚡ advanced with NO user message")

        if event.is_ready_for_input():
            # NOTE: these are INTERNAL ids ("0_companion_labpoll"), not the configured
            # thread_id. Match on the suffix, not equality.
            print(f"   … background: {getattr(event, 'active_companion_thread_ids', None)}")

        if event.is_terminal():
            print(f"\n✅ {event.type}")
            break

## 5. Reading the timeline by thread

Every event carries `thread_reference_id`: `"0"` for the main thread, or the companion's
configured id. It is a **computed** field — derived from the internal thread id rather than
stored — so it is present even on events written before the feature existed.

In [ ]:
from collections import Counter

by_thread = Counter(t for t, _ in timeline)
print("events per thread:")
for thread, n in by_thread.most_common():
    print(f"  {str(thread):12s} {n}")

print("\ncompanion-specific event types seen:")
for t in ("companion_edge", "companion_step_boundary", "waiting_evaluation_boundary",
          "waiting_condition_matched", "self_loop_exhausted", "workflow_ready_for_input"):
    seen = sum(1 for _, ty in timeline if ty == t)
    print(f"  {t:30s} {seen}")

## 6. Over REST — and the server build it needs

`client.runs.execute()` does not advance background work on its own:

- `pump_companions()` advances due background work by one step, without consuming a user turn;
- `drive_background_work()` polls it until nothing is left, **bounded** by `max_iterations` so a
  long-running companion cannot block forever;
- gate both on `response.has_background_work`.

> **⚠️ Needs a server build that includes `interactly-ai@1c41c4c10`.**
>
> Before that fix the REST path did not run companions at all. `execute()` returned as soon as the
> main thread parked, and the companion's first node emitted `start_node_run` and was then abandoned
> — no `end_node_run`, `has_background_work` stuck at `False`, nothing for `pump_companions()` to
> advance. The cause was the REST turn loop breaking out of the runtime generator on the busy-wait
> event, which cancelled the drain loop while the companion was still mid-execution.
>
> `stream()` was never affected: the WebSocket driver does not break on busy-wait.
>
> The cell below prints what the server actually returned, so you can tell which build you are on
> rather than taking either answer on trust.

In [ ]:
response = await client.runs.execute(WORKFLOW_ID, command=WorkflowCommand.START)
print(f"turn {response.turn_number}  status={response.status}")
print(f"  has_background_work   = {response.has_background_work}")
# `has_active_companions` is retained for backward compatibility and is a strict subset —
# it misses parked threads with armed waiting-evaluations. Poll on has_background_work.
print(f"  has_active_companions = {response.has_active_companions}   (narrower, legacy)")

companion_events = [
    (ev.get("type"), ev.get("origin_thread_id"))
    for ev in response.events
    if isinstance(ev, dict) and "companion" in str(ev.get("origin_thread_id") or "")
]
print("\ncompanion-thread events in this response:")
for kind, thread in companion_events:
    print(f"  {kind:28s} thread={thread}")

if response.has_background_work:
    pumped = await client.runs.drive_background_work(
        WORKFLOW_ID, response.run_id, interval_seconds=1.0, max_iterations=30)
    print(f"\npumped {len(pumped)} time(s)")
    for r in pumped:
        for ev in r.events:
            kind = ev.get("type") if isinstance(ev, dict) else getattr(ev, "type", "?")
            if kind in ("assistant_response", "waiting_condition_matched", "companion_step_boundary"):
                print("  ", kind)
    # Distinguishes "finished" from "ran out of iterations".
    if pumped and pumped[-1].has_background_work:
        print("⚠️  still pumping when max_iterations ran out")
else:
    # Either the companion already finished within the turn, or this server predates the fix.
    started = any(k == "start_node_run" for k, _ in companion_events)
    ended = any(k == "end_node_run" for k, _ in companion_events)
    if started and not ended:
        print("\n⚠️  the companion started and was never advanced — this server predates "
              "interactly-ai@1c41c4c10. Drive this workflow over stream() instead.")
    else:
        print("\nNo background work pending — nothing to pump.")

## Cleanup

In [ ]:
await client.workflows.delete(WORKFLOW_ID)
await client.close()
print("Deleted", WORKFLOW_ID)

## See also

- [`20_waiting_conditions.ipynb`](20_waiting_conditions.ipynb) — the edge that fires with no user message
- [`15_websocket_streaming_events.ipynb`](15_websocket_streaming_events.ipynb) — the event protocol
- [Companion threads guide](../docs/guides/companion_threads.md) · [Self-loops guide](../docs/guides/self_loops.md)

### Not the same "companion" as example 6

[`wf_example_progression_6`](../wf_examples/wf_example_progression_6.md) uses `CompanionEdgeConfig` —
an **edge type** pairing a Say agent with a silent Worker agent inside one turn.
`CompanionThreadConfig` is a **setting on a direct edge** that forks a genuinely concurrent
background thread. Same word, unrelated mechanisms.